# Contrastive Learning for Image Representation

Exploring triplet loss and SimCLR contrastive pretraining strategies to learn visual representations on CIFAR-10, with a custom Vision Transformer (ViT) implementation evaluated via linear probing.

In [ ]:
!pip install einops
!pip install datasets
!pip install umap-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import RandAugment
from IPython.core.debugger import set_trace

import transformers
from einops import rearrange
from einops.layers.torch import Rearrange
from datasets import load_dataset

import os
import sys
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

import umap

from google.colab import drive

torch.manual_seed(0)

In [ ]:
def triplet_loss(queries, keys, margin=0.5):
    queries = F.normalize(queries, dim=-1)
    keys = F.normalize(keys, dim=-1)
    sim = torch.matmul(queries, keys.T)
    pos = torch.diag(sim)
    mask = ~torch.eye(sim.shape[0], dtype=torch.bool, device=sim.device)
    neg = sim[mask].view(sim.shape[0], -1)
    loss = F.relu(neg - pos.unsqueeze(1) + margin)
    return loss.mean()


def nt_xent_loss(queries, keys, temperature=0.1):
    B = queries.shape[0]
    z = torch.cat([queries, keys], dim=0)
    z = F.normalize(z, dim=-1)
    sim = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, float('-inf'))
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, labels)

In [ ]:
NUM_EPOCHS = 10

## ResNet Backbone

A custom ResNet with residual blocks and a projection head is used as the encoder backbone. Features are extracted via average pooling before the projection head for downstream linear evaluation.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channel, interm_channel, out_channel, stride=1):
        """
        Inputs:
        in_channel = number of channels in the input to the first convolutional layer
        interm_channel = number of channels in the output of the first convolutional layer
                       = number of channels in the input to the second convolutional layer
        out_channel = number of channels in the output
        stride = stride for convolution, defaults to 1
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channel, interm_channel, kernel_size = 3, stride = stride, padding = 1)
        self.conv2 = nn.Conv2d(interm_channel, out_channel, kernel_size = 3, stride = stride, padding = 1)
        self.conv3 = nn.Conv2d(in_channel, out_channel, kernel_size = 1, stride = stride) # 1x1 convolution
        self.bn1 = nn.BatchNorm2d(interm_channel)
        self.bn2 = nn.BatchNorm2d(out_channel)

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        x = self.conv3(x)
        y +=  x # identity mapping
        return F.relu(y)


class ResNet(nn.Module):
    def __init__(self, num_blocks, layer1_channel, layer2_channel, out_channel):
        """
        Inputs:
        num_blocks = number of blocks in a block layer
        layer1_channel = number of channels in the input to the first block layer
        layer2_channel = number of channels in the output of the first block layer
                       = number of channels in the input to the second blcok layer
        out_channel = number of channels in the output
        """
        super(ResNet, self).__init__()
        self.first = nn.Sequential(
            nn.Conv2d(3, layer1_channel, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(layer1_channel), nn.SiLU(),
        )

        self.last = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),

        )

        self.layer1 = self.block_layer(num_blocks, layer1_channel, layer2_channel)
        self.layer2 = self.block_layer(num_blocks, layer2_channel, out_channel)

        self.projection_head = nn.Sequential(
            nn.Linear(out_channel, out_channel),
            nn.SiLU(),
            nn.Linear(out_channel, out_channel)
        )


    def block_layer(self, num_blocks, in_channel, out_channel):
        """
        Inputs:
        num_blocks = number of blocks in the block layer
        in_channel = number of input channels to the entire block layer
        out_channel = number of output channels in the output of the entire block layer
        """
        blk = []
        for i in range(num_blocks):
            if i == 0:
                blk.append(ResidualBlock(in_channel, out_channel, out_channel))
            else:
                blk.append(ResidualBlock(out_channel, out_channel, out_channel))

        return nn.Sequential(*blk)


    def forward(self, x, return_embedding=False):
        # x: (batch_size, 3, 32, 32)
        y = self.first(x)
        y = self.layer1(y)
        # 2x2 avg pooling
        y = F.avg_pool2d(y, kernel_size=2, stride=2)
        y = self.layer2(y)
        y = self.last(y)
        if return_embedding:
            return y
        y = self.projection_head(y)
        return y

CIFAR-10 is loaded from HuggingFace. Positive pairs are constructed by applying two independent random augmentations (random resized crop, horizontal flip, RandAugment) to the same image for contrastive training.

In [ ]:
# Load cifar10 data from Hugging Face
cifar10 = load_dataset('cifar10')

# Load the data
train_data = cifar10['train']
test_data = cifar10['test']

# Define randomized transforms for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    RandAugment(2, 9),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define fixed SimCLR transforms for test-time
test_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define the SimCLR dataset
class SimCLRDataset(Dataset):
    def __init__(self, data, transform, split='train'):
        self.split = split
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]['img']
        if self.split == 'train':
            image1 = self.transform(image)
            image2 = self.transform(image)
            return image1, image2
        else:
            image = self.transform(image)
            return image

# Create the SimCLR dataset
train_dataset = SimCLRDataset(train_data, train_transform, split='train')
val_dataset = SimCLRDataset(train_data, test_transform, split='test')
test_dataset = SimCLRDataset(test_data, test_transform, split='test')

# Create the SimCLR dataloaders
BATCH_SIZE = 256
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### Triplet Loss Training

The ResNet is trained with triplet loss, treating augmented views of the same image as positives and all other images in the batch as negatives.

In [ ]:
# Define the triplet model
triplet_model = ResNet(2, 64, 128, 256).cuda()

# Define the optimizer
optimizer = optim.AdamW(triplet_model.parameters(), lr=0.001)

# Define the learning rate scheduler
scheduler = transformers.get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=NUM_EPOCHS * len(train_dataloader))

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total_loss = 0
    for image1, image2 in tqdm(loader):
        image1 = image1.to(device)
        image2 = image2.to(device)

        queries = model(image1)
        keys = model(image2)

        loss = loss_fn(queries, keys)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# Train the model
loss_fn = triplet_loss
for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(triplet_model, train_dataloader, optimizer, scheduler, loss_fn)
    print(f'Epoch {epoch}, Train Loss: {train_loss:.4f}')

# Test the model by extracting features and training a linear classifier
triplet_model.eval()

In [ ]:
def extract_features(model, val_dataloader):
    features = []
    pixels = []
    with torch.no_grad():
        for image in val_dataloader:
            image = image.cuda()
            pixels.append(image.to('cpu'))
            feature = model(image, return_embedding=True)
            features.append(feature)
    features = torch.cat(features).to('cpu').numpy()
    return features, pixels

features, train_img_features = extract_features(triplet_model, val_dataloader)

In [ ]:
subsample = np.random.choice(features.shape[0], size=5000, replace=False)
features_subsample = features[subsample]
train_data_label = np.array(train_data['label'])[subsample]

classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=100, solver='saga', multi_class='multinomial', verbose=1))
classifier.fit(features_subsample, train_data_label)

In [ ]:
# Train the pixel-space classifier
train_img_array = np.concatenate(train_img_features, axis=0)

train_img_array = train_img_array[subsample]
pixel_classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=100, solver='saga', multi_class='multinomial', verbose=1))
pixel_classifier.fit(train_img_array.reshape(-1, 32 * 32 * 3), train_data_label)

In [ ]:
test_features, test_img_features = extract_features(triplet_model, test_dataloader)

In [ ]:
# Test the feature classifier
predictions = classifier.predict(test_features)
triplet_accuracy = accuracy_score(test_data['label'], predictions)
print(f'Accuracy: {triplet_accuracy:.4f}')

# Convert the 'img' list into a numpy array
test_img_array = np.concatenate(test_img_features, axis=0)

# Test the pixel-space classifier
pixel_predictions = pixel_classifier.predict(test_img_array.reshape(-1, 32 * 32 * 3))
pixel_accuracy = accuracy_score(test_data['label'], pixel_predictions)
print(f'Pixel Accuracy: {pixel_accuracy:.4f}')

In [ ]:
# Define the SimCLR model
simclr_model = ResNet(2, 64, 128, 256)
simclr_model = simclr_model.cuda()

simclr_model = simclr_model

# Define the optimizer
optimizer = optim.AdamW(simclr_model.parameters(), lr=0.001)

# Define the learning rate scheduler
scheduler = transformers.get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=NUM_EPOCHS * len(train_dataloader))

In [ ]:
# Train the model
loss_fn = nt_xent_loss
for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(simclr_model, train_dataloader, optimizer, scheduler, loss_fn)
    print(f'Epoch {epoch}, Train Loss: {train_loss:.4f}')

In [ ]:
# Test the model by extracting features and training a linear classifier
simclr_model.eval()

features, train_img_features = extract_features(simclr_model, val_dataloader)
test_features, test_img_features = extract_features(simclr_model, test_dataloader)

features_subsample = features[subsample]
train_data_label = np.array(train_data['label'])[subsample]

classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=100, solver='saga', multi_class='multinomial', verbose=1))
classifier.fit(features_subsample, train_data_label)

In [ ]:
predictions = classifier.predict(test_features)
simclr_accuracy = accuracy_score(test_data['label'], predictions)
print(f'Accuracy: {simclr_accuracy:.4f}')

In [ ]:
def posemb_sincos_2d(h, w, dim, temperature: int = 10000, dtype = torch.float32):
    '''
    h: Height of the patch.
    w: Width of the patch.
    dim: The dimension of the model embeddings.
    '''

    y, x = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
    assert (dim % 4) == 0, "feature dimension must be multiple of 4 for sincos emb"

    omega = torch.arange(dim // 4) / (dim // 4 - 1)
    omega = 1.0 / (temperature ** omega)

    y = y.flatten()[:, None] * omega[None, :]
    x = x.flatten()[:, None] * omega[None, :]
    pe = torch.cat((x.sin(), x.cos(), y.sin(), y.cos()), dim=1)
    return pe.type(dtype)

In [ ]:
simclr_vit = ViT(256, 4)
simclr_vit = simclr_vit.cuda()

# Define the optimizer
optimizer = optim.AdamW(simclr_vit.parameters(), lr=0.001)

# Define the learning rate scheduler
scheduler = transformers.get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=NUM_EPOCHS * len(train_dataloader))

# Train the model
loss_fn = nt_xent_loss
for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(simclr_vit, train_dataloader, optimizer, scheduler, loss_fn)
    print(f'Epoch {epoch}, Train Loss: {train_loss:.4f}')

In [ ]:
# Test the model by extracting features and training a linear classifier
simclr_vit.eval()

features, train_img_features = extract_features(simclr_vit, val_dataloader)
test_features, test_img_features = extract_features(simclr_vit, test_dataloader)

subsample = np.random.choice(features.shape[0], size=5000, replace=False)

features_subsample = features[subsample]
train_data_label = np.array(train_data['label'])[subsample]

classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=100, solver='saga', multi_class='multinomial', verbose=1))
classifier.fit(features_subsample, train_data_label)

In [ ]:
predictions = classifier.predict(test_features)
simclr_vit_accuracy = accuracy_score(test_data['label'], predictions)
print(f'Accuracy: {simclr_vit_accuracy:.4f}')

In [ ]:
# Visualize the features
class2name_list = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Reduce the dimensionality of the features
reducer = umap.UMAP()
scaled_test_features = StandardScaler().fit_transform(test_features)

reduced_features = reducer.fit_transform(scaled_test_features)

# Plot the features
plt.figure(figsize=(10, 10))
for i in range(10):
    test_label_array = np.array(test_data['label'])
    mask = test_label_array == i
    subsample = np.random.choice(np.where(mask)[0], size=100, replace=False)
    plt.scatter(reduced_features[subsample, 0], reduced_features[subsample, 1], label=class2name_list[i])
plt.title('Contrastive Feature Visualization')
plt.legend()
plt.show()

In [ ]:
# Visualize the pixel space
# Reduce the dimensionality of the pixel space
reducer = umap.UMAP()
test_img_array = np.concatenate(test_img_features, axis=0)
reduced_pixels = reducer.fit_transform(test_img_array.reshape(-1, 32 * 32 * 3))

# Plot the pixel space
plt.figure(figsize=(10, 10))
for i in range(10):
    mask = test_label_array == i
    subsample = np.random.choice(np.where(mask)[0], size=100, replace=False)
    plt.scatter(reduced_pixels[subsample, 0], reduced_pixels[subsample, 1], label=class2name_list[i])
plt.title('Pixel-Space Feature Visualization')
plt.legend()
plt.show()